# Assignment 1 — Build a Custom Missing-Value Imputer
**Feature Engineering & MLOps · Unit 1, Session 4 follow-up (Missing Values)**

This notebook implements a scikit-learn-compatible `CustomImputer` class from scratch, applies it to the real PrepEdge coaching-institute dataset (`student_performance_raw.csv`), verifies correctness against `sklearn.impute.SimpleImputer`, answers the reflection questions, and implements the bonus per-column strategy override.

**Sections:**
1. Imports
2. Load dataset
3. `CustomImputer` class definition
4. Train/test split
5. Fit on training data only
6. Transform train and test
7. Guard rail (transform before fit)
8. Verification — no missing values remain
9. Before/after mean & std comparison
10. Sanity check against `SimpleImputer`
11. Reflection questions
12. Bonus — per-column strategy override


## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted
import pandas.api.types as ptypes

pd.set_option("display.width", 120)
print("Libraries imported.")
print("numpy:", np.__version__, "| pandas:", pd.__version__)


Libraries imported.
numpy: 2.4.4 | pandas: 3.0.2


## 2. Load the Dataset

Loading `data/raw/student_performance_raw.csv` — the real PrepEdge coaching-institute dataset from Unit 1. It contains missingness under the three mechanisms studied in Session 4, plus a categorical and a free-text column:
- `weekly_study_hours` → **MCAR** (missing completely at random)
- `prev_exam_score` → **MAR** (linked to `attendance_pct`)
- `mock_test_3` → **MNAR** (linked to its own low values)
- `income_bracket` → missing categorical values
- `feedback_text` → missing free-text values

The other columns (`city`, `city_tier`, `course`, `batch_type`, `age`, `enrollment_date`, `mock_test_1`, `mock_test_2`, `doubt_sessions_attended`, `final_score`) are fully observed.

In [2]:
df = pd.read_csv("data/raw/student_performance_raw.csv")
print("Shape:", df.shape)
print()
print("Missing values per column:")
print(df.isna().sum())


Shape: (600, 17)

Missing values per column:
student_id                  0
city                        0
city_tier                   0
course                      0
batch_type                  0
age                         0
enrollment_date             0
attendance_pct              0
weekly_study_hours         36
income_bracket             30
prev_exam_score            25
mock_test_1                 0
mock_test_2                 0
mock_test_3                21
doubt_sessions_attended     0
feedback_text              68
final_score                 0
dtype: int64


In [3]:
df.head()


 student_id   city  city_tier       course batch_type  age enrollment_date  attendance_pct  weekly_study_hours income_bracket  prev_exam_score  mock_test_1  mock_test_2  mock_test_3  doubt_sessions_attended                               feedback_text  final_score
       1447  Patna          2     JEE Main    Weekday   18      2026-05-27            91.1                 0.8            <5L             65.2         68.1         59.0         72.2                        1   Need more practice sheets for weak topics         42.0
       1405  Patna          2         NEET    Weekday   16      2024-07-08            91.9                 2.4            NaN             64.9         82.0         88.4        100.0                        3        Would like more one-on-one mentoring         62.0
       1510 Mumbai          1     JEE Main    Weekday   18      2025-03-17            67.1                 6.7          5-10L             90.1         86.5         91.1         76.4                        5 G

student_id,city,city_tier,course,batch_type,age,enrollment_date,attendance_pct,weekly_study_hours,income_bracket,prev_exam_score,mock_test_1,mock_test_2,mock_test_3,doubt_sessions_attended,feedback_text,final_score
1447,Patna,2,JEE Main,Weekday,18,2026-05-27,91.1,0.8,<5L,65.2,68.1,59.0,72.2,1,Need more practice sheets for weak topics,42.0
1405,Patna,2,NEET,Weekday,16,2024-07-08,91.9,2.4,NaN,64.9,82.0,88.4,100.0,3,Would like more one-on-one mentoring,62.0
1510,Mumbai,1,JEE Main,Weekday,18,2025-03-17,67.1,6.7,5-10L,90.1,86.5,91.1,76.4,5,"Great teaching pace, doubts cleared quickly",56.4
1456,Delhi,1,NEET,Weekday,18,2025-06-16,70.7,2.0,10-20L,29.7,19.4,24.9,NaN,3,Need more practice sheets for weak topics,31.7
1202,Patna,2,JEE Advanced,Weekday,17,2026-04-16,69.3,6.5,>20L,70.0,63.8,74.5,69.1,4,Need more practice sheets for weak topics,44.5


## 3. `CustomImputer` Class Definition

Inherits from `BaseEstimator` and `TransformerMixin` so it plugs directly into scikit-learn pipelines. Column type is auto-detected with `pandas.api.types`, fitted statistics are stored with a trailing underscore (`fill_values_`) per scikit-learn convention, `transform()` never recomputes statistics, and calling `transform()` before `fit()` raises a clear error via `check_is_fitted`. The optional `column_overrides` parameter implements the **bonus** per-column strategy override (Section 12).

In [4]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """A scikit-learn-compatible imputer that fills missing values in both
    numeric and categorical columns, and optionally adds a binary
    '<column>_was_missing' indicator column for every column that had
    missing values in the training data.

    Column type is auto-detected (numeric vs. categorical) using
    pandas.api.types, so the caller does not need to specify column types
    manually.

    Parameters
    ----------
    numeric_strategy : {'mean', 'median'}, default='median'
        Statistic used to fill missing values in numeric columns.
    categorical_strategy : {'most_frequent'}, default='most_frequent'
        Strategy used to fill missing values in categorical / object columns.
        Currently only 'most_frequent' (the mode) is supported.
    add_missing_indicator : bool, default=True
        If True, adds one new binary column per column that had at least
        one missing value in the TRAINING data, named '<column>_was_missing'.
    column_overrides : dict, optional
        Optional per-column strategy override, e.g.
        {'weekly_study_hours': 'mean', 'income_bracket': 'most_frequent'}.
        Keys are column names, values are the strategy to use for that
        specific column (overrides the dataset-wide default for that column
        only). [Bonus feature]

    Attributes
    ----------
    fill_values_ : dict
        Mapping of column name -> fitted fill value, learned during fit().
    indicator_columns_ : list of str
        Names of columns that had missing values in the training data (and
        therefore get a '_was_missing' indicator column during transform).
    numeric_columns_ : list of str
        Columns detected as numeric during fit().
    categorical_columns_ : list of str
        Columns detected as categorical during fit().
    """

    def __init__(self, numeric_strategy="median", categorical_strategy="most_frequent",
                 add_missing_indicator=True, column_overrides=None):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _strategy_for(self, column, is_numeric):
        """Resolve the strategy to use for a given column, respecting
        column_overrides if provided, otherwise falling back to the
        dataset-wide default strategy for that column's type."""
        overrides = self.column_overrides or {}
        if column in overrides:
            return overrides[column]
        return self.numeric_strategy if is_numeric else self.categorical_strategy

    def _compute_fill_value(self, series, strategy):
        """Compute a single fill value for one column given a strategy."""
        if strategy == "mean":
            return series.mean()
        elif strategy == "median":
            return series.median()
        elif strategy == "most_frequent":
            mode = series.mode(dropna=True)
            return mode.iloc[0] if len(mode) > 0 else None
        else:
            raise ValueError(f"Unknown strategy '{strategy}'")

    def fit(self, X, y=None):
        """Learn the fill value for every column in X.

        For each column, detects whether it is numeric or categorical and
        computes the appropriate fill statistic (mean/median for numeric,
        mode for categorical), storing it in self.fill_values_. Also
        records which columns had missing values, for the missing
        indicator feature, and which columns were numeric vs categorical.

        Parameters
        ----------
        X : pandas.DataFrame
            Training data. Statistics are learned ONLY from this data.
        y : ignored
            Present for scikit-learn API compatibility.

        Returns
        -------
        self : CustomImputer
            The fitted imputer.
        """
        X = pd.DataFrame(X).copy()

        self.fill_values_ = {}
        self.numeric_columns_ = []
        self.categorical_columns_ = []
        self.indicator_columns_ = []

        for column in X.columns:
            series = X[column]
            is_numeric = ptypes.is_numeric_dtype(series)

            if is_numeric:
                self.numeric_columns_.append(column)
            else:
                self.categorical_columns_.append(column)

            strategy = self._strategy_for(column, is_numeric)
            self.fill_values_[column] = self._compute_fill_value(series, strategy)

            if series.isna().any():
                self.indicator_columns_.append(column)

        self.feature_names_in_ = list(X.columns)
        return self

    def transform(self, X):
        """Return a COPY of X with missing values filled using the
        statistics learned in fit() (nothing is recomputed here), plus
        optional '<column>_was_missing' indicator columns.

        Parameters
        ----------
        X : pandas.DataFrame
            Data to transform (can be the training data, test data, or any
            new data with the same columns as the data fit() was called on).

        Returns
        -------
        pandas.DataFrame
            A new DataFrame (X is not modified in place) with missing
            values filled in, and indicator columns appended if
            add_missing_indicator=True.
        """
        check_is_fitted(self, attributes=["fill_values_"])
        X = pd.DataFrame(X).copy()

        if self.add_missing_indicator:
            for column in self.indicator_columns_:
                if column in X.columns:
                    X[f"{column}_was_missing"] = X[column].isna().astype(int)

        for column, fill_value in self.fill_values_.items():
            if column in X.columns:
                X[column] = X[column].fillna(fill_value)

        return X

print("CustomImputer class defined.")


CustomImputer class defined.


## 4. Train/Test Split (80/20)

In [5]:
feature_cols = [c for c in df.columns if c != "student_id"]
X = df[feature_cols]

X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


Train shape: (480, 16)
Test shape : (120, 16)


## 5. Fit `CustomImputer` on the TRAINING Split Only

This is the critical train/test-discipline step: `fit()` only ever sees `X_train`. The test split is never used to compute fill statistics — see Reflection Question 1.

In [6]:
imputer = CustomImputer(numeric_strategy="median", categorical_strategy="most_frequent",
                         add_missing_indicator=True)
imputer.fit(X_train)

print("Fitted fill values:")
for col, val in imputer.fill_values_.items():
    print(f"  {col:24s} -> {val}")

print()
print("Numeric columns    :", imputer.numeric_columns_)
print("Categorical columns:", imputer.categorical_columns_)
print("Indicator columns  :", imputer.indicator_columns_)


Fitted fill values:
  city                     -> Mumbai
  city_tier                -> 1.0
  course                   -> NEET
  batch_type               -> Weekday
  age                      -> 17.0
  enrollment_date          -> 2024-10-23
  attendance_pct           -> 78.35
  weekly_study_hours       -> 5.0
  income_bracket           -> 5-10L
  prev_exam_score          -> 66.1
  mock_test_1              -> 66.55
  mock_test_2              -> 66.35
  mock_test_3              -> 71.3
  doubt_sessions_attended  -> 4.0
  feedback_text            -> Need more practice sheets for weak topics
  final_score              -> 52.7

Numeric columns    : ['city_tier', 'age', 'attendance_pct', 'weekly_study_hours', 'prev_exam_score', 'mock_test_1', 'mock_test_2', 'mock_test_3', 'doubt_sessions_attended', 'final_score']
Categorical columns: ['city', 'course', 'batch_type', 'enrollment_date', 'income_bracket', 'feedback_text']
Indicator columns  : ['weekly_study_hours', 'income_bracket', 'prev_exam_s

## 6. Transform Both Train and Test with the Fitted Imputer

In [7]:
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Imputed train shape:", X_train_imputed.shape)
print("Imputed test shape :", X_test_imputed.shape)
print()
print("New indicator columns added:")
print([c for c in X_train_imputed.columns if c.endswith("_was_missing")])


Imputed train shape: (480, 21)
Imputed test shape : (120, 21)

New indicator columns added:
['weekly_study_hours_was_missing', 'income_bracket_was_missing', 'prev_exam_score_was_missing', 'mock_test_3_was_missing', 'feedback_text_was_missing']


In [8]:
X_train_imputed.head()


     city  city_tier     course batch_type  age enrollment_date  attendance_pct  weekly_study_hours income_bracket  prev_exam_score  mock_test_1  mock_test_2  mock_test_3  doubt_sessions_attended                               feedback_text  final_score  weekly_study_hours_was_missing  income_bracket_was_missing  prev_exam_score_was_missing  mock_test_3_was_missing  feedback_text_was_missing
     Pune          1   JEE Main    Weekend   17      2024-12-17            59.8                 2.0          5-10L             56.5         58.3         46.2         54.5                        5        Would like more one-on-one mentoring         39.3                               0                           0                            0                        0                          0
     Kota          2   JEE Main    Weekday   15      2025-08-19            72.9                 5.0         10-20L             57.8         46.6         40.9         34.9                        6   Need more prac

city,city_tier,course,batch_type,age,enrollment_date,attendance_pct,weekly_study_hours,income_bracket,prev_exam_score,mock_test_1,mock_test_2,mock_test_3,doubt_sessions_attended,feedback_text,final_score,weekly_study_hours_was_missing,income_bracket_was_missing,prev_exam_score_was_missing,mock_test_3_was_missing,feedback_text_was_missing
Pune,1,JEE Main,Weekend,17,2024-12-17,59.8,2.0,5-10L,56.5,58.3,46.2,54.5,5,Would like more one-on-one mentoring,39.3,0,0,0,0,0
Kota,2,JEE Main,Weekday,15,2025-08-19,72.9,5.0,10-20L,57.8,46.6,40.9,34.9,6,Need more practice sheets for weak topics,43.2,1,0,0,0,0
Hyderabad,2,Foundation,Weekday,17,2026-02-14,72.1,6.2,5-10L,45.1,52.0,59.1,65.9,1,Need more practice sheets for weak topics,44.0,0,0,0,0,0
Delhi,1,NEET,Weekday,19,2025-10-04,82.5,1.1,5-10L,54.3,58.7,63.8,74.0,4,Mock tests really helped identify gaps,39.6,0,0,0,0,0
Lucknow,2,NEET,Weekend,15,2025-05-29,98.0,2.8,<5L,78.8,73.8,78.6,87.9,3,"Great teaching pace, doubts cleared quickly",55.0,0,0,0,0,0


## 7. Guard Rail — `transform()` Before `fit()` Must Raise a Clear Error

In [9]:
try:
    CustomImputer().transform(X_test)
except Exception as e:
    print(f"Raised as expected -> {type(e).__name__}: {e}")


Raised as expected -> NotFittedError: This CustomImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


## 8. Verify Programmatically That No Missing Values Remain

In [10]:
orig_cols = [c for c in feature_cols]
train_missing_after = X_train_imputed[orig_cols].isna().sum().sum()
test_missing_after = X_test_imputed[orig_cols].isna().sum().sum()

assert train_missing_after == 0, "Missing values remain in imputed train set!"
assert test_missing_after == 0, "Missing values remain in imputed test set!"

print("Missing values remaining in TRAIN (original columns):", train_missing_after)
print("Missing values remaining in TEST  (original columns):", test_missing_after)
print("PASSED: no missing values remain after imputation.")


Missing values remaining in TRAIN (original columns): 0
Missing values remaining in TEST  (original columns): 0
PASSED: no missing values remain after imputation.


## 9. Mean & Std of Each Numeric Column — Before vs. After Imputation

In [11]:
numeric_cols = imputer.numeric_columns_

rows = []
for col in numeric_cols:
    before_mean = X_train[col].mean()
    before_std = X_train[col].std()
    after_mean = X_train_imputed[col].mean()
    after_std = X_train_imputed[col].std()
    rows.append({
        "column": col,
        "mean_before": round(before_mean, 3),
        "mean_after": round(after_mean, 3),
        "std_before": round(before_std, 3),
        "std_after": round(after_std, 3),
    })

stats_table = pd.DataFrame(rows)
print(stats_table.to_string(index=False))


                 column  mean_before  mean_after  std_before  std_after
              city_tier        1.402       1.402       0.491      0.491
                    age       16.956      16.956       1.731      1.731
         attendance_pct       78.546      78.546      13.460     13.460
     weekly_study_hours        6.064       6.009       4.350      4.242
        prev_exam_score       65.825      65.838      14.371     14.022
            mock_test_1       65.945      65.945      16.566     16.566
            mock_test_2       67.886      67.886      18.109     18.109
            mock_test_3       70.702      70.720      19.077     18.777
doubt_sessions_attended        4.004       4.004       2.026      2.026
            final_score       53.087      53.087      10.458     10.458


**Comment:** the mean barely moves for any imputed column, since median/mean imputation is designed to preserve the central tendency of the observed data by construction — `weekly_study_hours`, `prev_exam_score`, and `mock_test_3` all show mean shifts under 0.1. The standard deviation, however, **shrinks slightly** for every imputed column (e.g. `weekly_study_hours` drops from 4.35 → 4.24, `mock_test_3` from 19.08 → 18.78): filling in a constant value for every missing row adds no new variance, so the imputed columns look artificially more consistent than the true underlying data would be. This is the classic "imputation flattens variance" effect from Session 4, and it's most visible on `mock_test_3` and `prev_exam_score`, the two columns with the highest missingness rates. Fully-observed numeric columns like `mock_test_1`, `mock_test_2`, `age`, and `final_score` are unaffected, as expected.

## 10. Sanity Check — Compare Fill Values Against `sklearn.impute.SimpleImputer`

In [12]:
sklearn_median_imputer = SimpleImputer(strategy="median")
sklearn_median_imputer.fit(X_train[numeric_cols])

match_report = []
for i, col in enumerate(numeric_cols):
    custom_val = imputer.fill_values_[col]
    sklearn_val = sklearn_median_imputer.statistics_[i]
    match = np.isclose(custom_val, sklearn_val)
    match_report.append({"column": col, "custom_fill": custom_val,
                          "sklearn_fill": sklearn_val, "match": match})

match_df = pd.DataFrame(match_report)
print(match_df.to_string(index=False))
print()
all_match = match_df["match"].all()
assert all_match, "Custom imputer fill values do not match sklearn's SimpleImputer!"
print("PASSED: CustomImputer's fill values match sklearn's SimpleImputer exactly.")


                 column  custom_fill  sklearn_fill  match
              city_tier         1.00          1.00   True
                    age        17.00         17.00   True
         attendance_pct        78.35         78.35   True
     weekly_study_hours         5.00          5.00   True
        prev_exam_score        66.10         66.10   True
            mock_test_1        66.55         66.55   True
            mock_test_2        66.35         66.35   True
            mock_test_3        71.30         71.30   True
doubt_sessions_attended         4.00          4.00   True
            final_score        52.70         52.70   True

PASSED: CustomImputer's fill values match sklearn's SimpleImputer exactly.


## 11. Reflection Questions

### 1. Why must `fit()` be called only on the training split, and never on the full dataset or the test split?

If `fit()` sees the test split, the fill statistics (mean/median/mode) are computed partly from data the model is supposed to have never seen — this is **data leakage**. The test set is meant to simulate genuinely new, unseen data, and in a real deployment the model would never get to peek at future data before making a prediction. Fitting on the full dataset makes the reported test performance overly optimistic and unreliable, because information from the test rows (their contribution to the mean/median) has silently leaked into the values used to fill the training set too. Fitting on the training split only keeps the evaluation honest and mirrors how the pipeline would actually be used in production, where a new row's missing values must be filled using statistics learned in the past.

### 2. `mock_test_3` is MNAR. Does mean/median imputation genuinely solve the problem? What does `add_missing_indicator` contribute?

No — mean/median imputation does not genuinely solve the problem for `mock_test_3`. Because the missingness itself is **informative** (low scorers are disproportionately likely to be missing), filling every gap with the overall median (71.3 in our fitted imputer) systematically overestimates the true values for the missing rows and biases the column's distribution upward, understating how many students actually struggled. Plain imputation also destroys the very signal that "this student's result went missing" carries — which, under MNAR, is itself predictive information. The `add_missing_indicator` feature recovers exactly that signal: by adding a `mock_test_3_was_missing` binary flag (which our fitted imputer does add — see Section 6), a downstream model can learn that missingness on this column correlates with weaker performance, effectively letting the model separate "low but observed" from "probably low and unobserved" students, even though the imputed numeric value alone can't tell them apart.

### 3. A brand-new column, entirely missing in training but present in test — what happens now, and what should a production version do?

As implemented, `fit()` never sees this column (it wasn't in `X_train`), so `self.fill_values_` has no entry for it. In `transform()`, the loop `for column, fill_value in self.fill_values_.items()` only fills columns it has a stored value for, so the new column would simply be **left untouched — passed through with its missing values still present** (or entirely missing, since nothing was ever learned for it). This would silently defeat the purpose of the imputer for that column and could break a downstream model expecting no NaNs. A production-grade version should instead: (a) explicitly detect columns present in `transform(X)` but absent from `feature_names_in_` and either raise a clear warning/error or drop them, since the imputer has no learned statistic to use safely; and (b) for a column that existed in training but was **100% missing there**, fall back to a sensible global default (e.g. 0 for numeric, a dedicated `"Unknown"` category, or refuse to fit with a clear error) rather than silently storing `NaN`/`None` as the "fill value" and propagating missingness downstream.

## 12. Bonus — Per-Column Strategy Override (`column_overrides`)

Demonstrating `column_overrides` on two columns with different strategies: `weekly_study_hours` uses `'mean'` instead of the dataset-wide `'median'` default, and `mock_test_3` also uses `'mean'` instead of `'median'`.

In [13]:
bonus_imputer = CustomImputer(
    numeric_strategy="median",
    categorical_strategy="most_frequent",
    add_missing_indicator=True,
    column_overrides={
        "weekly_study_hours": "mean",
        "mock_test_3": "mean",
    },
)
bonus_imputer.fit(X_train)

print("Default-strategy fill value (median) for weekly_study_hours:",
      round(imputer.fill_values_["weekly_study_hours"], 3))
print("Override-strategy fill value (mean) for weekly_study_hours  :",
      round(bonus_imputer.fill_values_["weekly_study_hours"], 3))
print()
print("Default-strategy fill value (median) for mock_test_3:",
      round(imputer.fill_values_["mock_test_3"], 3))
print("Override-strategy fill value (mean) for mock_test_3  :",
      round(bonus_imputer.fill_values_["mock_test_3"], 3))
print()

manual_mean_study = X_train["weekly_study_hours"].mean()
manual_mean_mock3 = X_train["mock_test_3"].mean()
assert np.isclose(bonus_imputer.fill_values_["weekly_study_hours"], manual_mean_study)
assert np.isclose(bonus_imputer.fill_values_["mock_test_3"], manual_mean_mock3)
print("PASSED: column_overrides correctly applies a different strategy per column.")

bonus_transformed = bonus_imputer.transform(X_test)
print()
print("Bonus-imputed test set — no missing values in overridden columns:")
print(bonus_transformed[["weekly_study_hours", "mock_test_3"]].isna().sum())


Default-strategy fill value (median) for weekly_study_hours: 5.0
Override-strategy fill value (mean) for weekly_study_hours  : 6.064

Default-strategy fill value (median) for mock_test_3: 71.3
Override-strategy fill value (mean) for mock_test_3  : 70.702

PASSED: column_overrides correctly applies a different strategy per column.

Bonus-imputed test set — no missing values in overridden columns:
weekly_study_hours    0
mock_test_3           0
dtype: int64
